In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt #绘图工具
import seaborn as sns
import warnings
import random
import time
import torch
from sklearn.utils import shuffle
warnings.filterwarnings("ignore")
#设置jupyter显示多行结果
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all' #默认为'last'
# #显示所有列
pd.set_option('display.max_columns', None) #原来中间会有部分列的显示被省略
# #显示所有行
# pd.set_option('display.max_rows', None)
#设置value的显示长度为100，默认为50
pd.set_option('max_colwidth',100)
plt.rcParams['font.sans-serif'] = ['SimHei']  #显示中文
plt.rcParams['axes.unicode_minus']=False #用来正常显示负号

In [2]:
data = pd.read_csv("ml-100k.csv",sep="\t")
data

,user,item,rating,time,label,time_stamp
0,258,254,4,874724710,1,1997-09-20 03:05:10
1,258,285,4,874724727,1,1997-09-20 03:05:27
2,258,297,4,874724754,1,1997-09-20 03:05:54
3,258,184,4,874724781,1,1997-09-20 03:06:21
4,258,172,4,874724843,1,1997-09-20 03:07:23
...,...,...,...,...,...,...
99995,728,688,4,893286638,1,1998-04-22 23:10:38
99996,728,312,3,893286638,0,1998-04-22 23:10:38
99997,728,327,3,893286638,0,1998-04-22 23:10:38
99998,728,747,4,893286638,1,1998-04-22 23:10:38


In [3]:
def generate_candidate(row):
    can_tmp = [i for i in range(1682)]
    can_tmp.remove(row)
    candidate = [row] + (random.sample(list(can_tmp), 4))
    random.shuffle(candidate)
    
    return candidate

In [4]:
data["candidate"] = data["item"].apply(generate_candidate)
data

,user,item,rating,time,label,time_stamp,candidate
0,258,254,4,874724710,1,1997-09-20 03:05:10,"[254, 558, 1306, 1408, 567]"
1,258,285,4,874724727,1,1997-09-20 03:05:27,"[793, 285, 1262, 662, 29]"
2,258,297,4,874724754,1,1997-09-20 03:05:54,"[968, 297, 1628, 622, 1028]"
3,258,184,4,874724781,1,1997-09-20 03:06:21,"[902, 1102, 184, 180, 1067]"
4,258,172,4,874724843,1,1997-09-20 03:07:23,"[457, 357, 1523, 172, 844]"
...,...,...,...,...,...,...,...
99995,728,688,4,893286638,1,1998-04-22 23:10:38,"[1167, 688, 1276, 1385, 1625]"
99996,728,312,3,893286638,0,1998-04-22 23:10:38,"[312, 754, 1116, 1569, 1467]"
99997,728,327,3,893286638,0,1998-04-22 23:10:38,"[1605, 262, 1095, 1490, 327]"
99998,728,747,4,893286638,1,1998-04-22 23:10:38,"[728, 622, 5, 536, 747]"


In [5]:
data.to_csv("ml-100k_all_data.csv",index=False, sep="\t")
data[:50000].to_csv("ml-100k_part1.csv",index=False, sep="\t")
data[50000:].to_csv("ml-100k_part2.csv",index=False, sep="\t")

In [6]:
from sklearn.utils import shuffle

data_part1 = pd.read_csv("ml-100k_part1.csv",sep="\t")
data_part2 = pd.read_csv("ml-100k_part2.csv",sep="\t")
data_part1
data_part2

,user,item,rating,time,label,time_stamp,candidate
0,258,254,4,874724710,1,1997-09-20 03:05:10,"[254, 558, 1306, 1408, 567]"
1,258,285,4,874724727,1,1997-09-20 03:05:27,"[793, 285, 1262, 662, 29]"
2,258,297,4,874724754,1,1997-09-20 03:05:54,"[968, 297, 1628, 622, 1028]"
3,258,184,4,874724781,1,1997-09-20 03:06:21,"[902, 1102, 184, 180, 1067]"
4,258,172,4,874724843,1,1997-09-20 03:07:23,"[457, 357, 1523, 172, 844]"
...,...,...,...,...,...,...,...
49995,177,565,4,882826915,1,1997-12-22 21:41:55,"[565, 1059, 811, 127, 199]"
49996,177,650,4,882826915,1,1997-12-22 21:41:55,"[991, 650, 1567, 437, 515]"
49997,177,316,4,882826915,1,1997-12-22 21:41:55,"[316, 1666, 418, 259, 1028]"
49998,177,194,4,882826944,1,1997-12-22 21:42:24,"[698, 1222, 727, 1166, 194]"


,user,item,rating,time,label,time_stamp,candidate
0,177,97,5,882826944,1,1997-12-22 21:42:24,"[1641, 1092, 97, 1195, 1647]"
1,177,678,4,882826944,1,1997-12-22 21:42:24,"[138, 418, 1011, 153, 678]"
2,177,384,4,882826982,1,1997-12-22 21:43:02,"[379, 384, 1190, 1057, 118]"
3,177,317,5,882826982,1,1997-12-22 21:43:02,"[294, 317, 1619, 980, 527]"
4,177,133,3,882826983,0,1997-12-22 21:43:03,"[1501, 911, 133, 57, 957]"
...,...,...,...,...,...,...,...
49995,728,688,4,893286638,1,1998-04-22 23:10:38,"[1167, 688, 1276, 1385, 1625]"
49996,728,312,3,893286638,0,1998-04-22 23:10:38,"[312, 754, 1116, 1569, 1467]"
49997,728,327,3,893286638,0,1998-04-22 23:10:38,"[1605, 262, 1095, 1490, 327]"
49998,728,747,4,893286638,1,1998-04-22 23:10:38,"[728, 622, 5, 536, 747]"


In [7]:
data_part1_shuffle = shuffle(data_part1)
data_part2_shuffle = shuffle(data_part1)
data_part1_shuffle
data_part2_shuffle

,user,item,rating,time,label,time_stamp,candidate
25675,526,962,4,879456030,1,1997-11-13 21:20:30,"[1676, 1305, 701, 962, 1639]"
24101,803,661,4,879442413,1,1997-11-13 17:33:33,"[1667, 799, 661, 124, 803]"
30052,445,302,2,879787859,0,1997-11-17 17:30:59,"[140, 946, 16, 202, 302]"
49430,876,537,4,882676533,1,1997-12-21 03:55:33,"[537, 20, 1259, 1284, 877]"
46411,398,1278,3,882341625,0,1997-12-17 06:53:45,"[218, 1278, 551, 1315, 1197]"
...,...,...,...,...,...,...,...
5178,286,741,3,875334196,0,1997-09-27 04:23:16,"[1553, 137, 741, 465, 1645]"
14677,172,328,4,877557345,1,1997-10-22 21:55:45,"[328, 185, 301, 867, 901]"
29044,780,179,4,879633895,1,1997-11-15 22:44:55,"[41, 179, 153, 1420, 611]"
31439,325,173,4,879874825,1,1997-11-18 17:40:25,"[1150, 857, 582, 226, 173]"


,user,item,rating,time,label,time_stamp,candidate
6645,592,49,4,875660009,1,1997-09-30 22:53:29,"[49, 1354, 1198, 1047, 1493]"
23581,414,431,4,879439610,1,1997-11-13 16:46:50,"[501, 171, 400, 431, 115]"
17529,814,1038,5,878693870,1,1997-11-05 01:37:50,"[1169, 729, 456, 1038, 638]"
42983,912,21,5,881369920,1,1997-12-06 00:58:40,"[21, 98, 680, 1279, 1565]"
23769,320,31,3,879440716,0,1997-11-13 17:05:16,"[294, 338, 31, 77, 660]"
...,...,...,...,...,...,...,...
47878,12,176,5,882397271,1,1997-12-17 22:21:11,"[41, 176, 1480, 1018, 1192]"
10965,80,117,2,876533764,0,1997-10-11 01:36:04,"[763, 1020, 117, 331, 727]"
48511,151,548,4,882476261,1,1997-12-18 20:17:41,"[1318, 548, 999, 544, 309]"
12169,371,184,5,876869445,1,1997-10-14 22:50:45,"[184, 24, 851, 1311, 1173]"


In [8]:
data_part1_shuffle.to_csv("ml-100k_part1_shuffle.csv",index=False, sep="\t")
data_part2_shuffle.to_csv("ml-100k_part2_shuffle.csv",index=False, sep="\t")

In [9]:
np.random.randint(0, 1682, (99))

array([ 369, 1311,  378,  494,  193,  311, 1582, 1402, 1537,  836,  796,
        313,  615,  140,  250,  475,  667, 1616, 1068,  233,  205,  849,
        201, 1393,   20, 1614,  348, 1461,  521,  534,  414,  274,  239,
       1487,  949,  249, 1506, 1409,  314, 1085,  253, 1209,  160,  574,
       1287,  868,  914, 1254, 1664, 1275,  705,  496,   71,   57,  603,
        371, 1044,  378, 1483, 1665,  275,  421,  885,  151, 1464, 1218,
       1263,  457, 1671,  372,   16,  637,   79, 1257, 1392, 1669, 1256,
       1083,  223, 1667,  813,  626, 1656, 1428, 1026, 1251,  511, 1446,
        667,  634,  717,  547,    9, 1082,  390, 1607, 1083, 1101,  532])

In [10]:
np.arange(1682)

array([   0,    1,    2, ..., 1679, 1680, 1681])

In [11]:
generate_candidate(1)

[1, 1041, 296, 1364, 1626]